# Train "Hey Huncho" — v3 (official openWakeWord pipeline)

This uses **dscripka's own openWakeWord training pipeline** (the maintained, battle-tested one), not the LiveKit trainer that kept failing. The wake phrase is already hardcoded to **"hey huncho"** — nothing to edit.

**Before running:** Runtime → Change runtime type → **T4 GPU** → Save. Then Runtime → **Run all** (approve the Drive prompt when it appears).

Clip generation ~10 min, training ~15–25 min. The final `huncho.onnx` saves to your Drive **and** downloads to your computer. Keep the tab open.

In [ ]:
# 0) Mount Drive (safety net) + confirm GPU
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib
OUT = '/content/drive/MyDrive/huncho_wakeword'
pathlib.Path(OUT).mkdir(parents=True, exist_ok=True)
print('Drive output dir:', OUT)
!nvidia-smi -L

In [ ]:
# 1) Environment setup (verbatim from the official openWakeWord notebook)
# NOTE: pinned to v2.0.0 - rhasspy moved generate_samples.py out of the repo root on main,
# which breaks openWakeWord's train.py import. v2.0.0 has it and matches the model below.
!git clone -q -b v2.0.0 --depth 1 https://github.com/rhasspy/piper-sample-generator
!wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
!pip install piper-phonemize
!pip install webrtcvad

!git clone https://github.com/dscripka/openwakeword
!pip install -e ./openwakeword

!pip install mutagen==1.47.0
!pip install torchinfo==1.8.0
!pip install torchmetrics==1.2.0
!pip install speechbrain==0.5.14
!pip install audiomentations==0.33.0
!pip install torch-audiomentations==0.11.0
!pip install acoustics==0.2.6
!pip install tensorflow-cpu==2.8.1
!pip install tensorflow_probability==0.16.0
!pip install onnx_tf==1.10.0
!pip install pronouncing==0.2.0
!pip install datasets==2.14.6
!pip install deep-phonemizer==0.0.19

# Download shared feature models (Colab workaround)
import os
os.makedirs('./openwakeword/openwakeword/resources/models', exist_ok=True)
base = 'https://github.com/dscripka/openWakeWord/releases/download/v0.5.1'
for f in ['embedding_model.onnx','embedding_model.tflite','melspectrogram.onnx','melspectrogram.tflite']:
    !wget -q {base}/{f} -O ./openwakeword/openwakeword/resources/models/{f}
print('setup done')

In [ ]:
# 1b) Compatibility patch - openWakeWord deps call torchaudio.set_audio_backend,
#     removed in torchaudio 2.1+. Locate the file WITHOUT importing (the import is what crashes),
#     then neutralize the call so train.py imports cleanly on current Colab.
import importlib.util, pathlib, re
_spec = importlib.util.find_spec('torch_audiomentations')
_root = pathlib.Path(_spec.origin).parent
_n = 0
for _p in _root.rglob('*.py'):
    _t = _p.read_text()
    if 'set_audio_backend' in _t:
        _t = re.sub(r'^(\s*)torchaudio\.set_audio_backend\([^)]*\)',
                    r'\1pass  # patched: removed in torchaudio 2.1+', _t, flags=re.M)
        _p.write_text(_t)
        _n += 1
        print('patched', _p.name)
print('compat patch done -', _n, 'file(s)')

In [ ]:
# 2) Imports
import os, numpy as np, torch, sys
from pathlib import Path
import uuid, yaml, datasets, scipy
from tqdm import tqdm

In [ ]:
# 3) Room impulse responses (reverb augmentation)
output_dir = './mit_rirs'
os.makedirs(output_dir, exist_ok=True)
rir_dataset = datasets.load_dataset('davidscripka/MIT_environmental_impulse_responses', split='train', streaming=True)
for row in tqdm(rir_dataset):
    name = row['audio']['path'].split('/')[-1]
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

In [ ]:
# 4) Background noise (one AudioSet shard) + music (FMA)
os.makedirs('audioset', exist_ok=True)
fname = 'bal_train09.tar'
!wget -O audioset/{fname} https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/{fname}
!cd audioset && tar -xf {fname}

output_dir = './audioset_16k'
os.makedirs(output_dir, exist_ok=True)
audioset_dataset = datasets.Dataset.from_dict({'audio': [str(i) for i in Path('audioset/audio').glob('**/*.flac')]})
audioset_dataset = audioset_dataset.cast_column('audio', datasets.Audio(sampling_rate=16000))
for row in tqdm(audioset_dataset):
    name = row['audio']['path'].split('/')[-1].replace('.flac', '.wav')
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

output_dir = './fma'
os.makedirs(output_dir, exist_ok=True)
fma_dataset = datasets.load_dataset('rudraml/fma', name='small', split='train', streaming=True)
fma_dataset = iter(fma_dataset.cast_column('audio', datasets.Audio(sampling_rate=16000)))
n_hours = 1
for i in tqdm(range(n_hours*3600//30)):
    row = next(fma_dataset)
    name = row['audio']['path'].split('/')[-1].replace('.mp3', '.wav')
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
    i += 1
    if i == n_hours*3600//30:
        break

In [ ]:
# 5) Pre-computed openWakeWord negative features (speech + false-positive validation)
!wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
!wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy
print('features downloaded')

In [ ]:
# 6) Config — phrase HARDCODED to "hey huncho" (nothing to edit)
config = yaml.load(open('openwakeword/examples/custom_model.yml', 'r').read(), yaml.Loader)

config['target_phrase'] = ['hey huncho']
config['model_name'] = 'huncho'
config['n_samples'] = 1000
config['n_samples_val'] = 1000
config['steps'] = 10000
config['target_accuracy'] = 0.6
config['target_recall'] = 0.25

config['background_paths'] = ['./audioset_16k', './fma']
config['false_positive_validation_data_path'] = 'validation_set_features.npy'
config['feature_data_files'] = {'ACAV100M_sample': 'openwakeword_features_ACAV100M_2000_hrs_16bit.npy'}

with open('my_model.yaml', 'w') as file:
    yaml.dump(config, file)
print('target_phrase:', config['target_phrase'], '| model_name:', config['model_name'])

In [ ]:
# 7) Step 1 — generate synthetic 'hey huncho' clips (~10 min). Safe to re-run if it stops early.
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --generate_clips

In [ ]:
# 8) Step 2 — augment the clips (noise, reverb, gain)
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips

In [ ]:
# 9) Step 3 — train the model (prints climbing step counts = working)
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model

In [ ]:
# 10) Save huncho.onnx to Drive AND download it
import glob, shutil
hits = glob.glob('my_custom_model/**/*.onnx', recursive=True) + glob.glob('**/huncho.onnx', recursive=True)
print('found onnx:', hits)
assert hits, 'No .onnx produced — check the training cell output above for errors.'
src = [h for h in hits if 'huncho' in h.lower()][0]
shutil.copy(src, 'huncho.onnx')
shutil.copy(src, OUT + '/huncho.onnx')
print('saved to Drive:', OUT + '/huncho.onnx')
from google.colab import files
files.download('huncho.onnx')